In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
from pathlib import Path

DATA_ROOT = Path("Data")
from read.elsevier import ElsevierParser  # ✅ 确保 read/elsevier.py 里定义了 ElsevierParser


def parse_elsevier_article(filepath, output_folder):
    """解析单篇 Elsevier HTML/XML 文件并将 paragraph 写入 txt"""
    try:
        # 1. 初始化解析器
        parser = ElsevierParser(filepath)

        # 2. 解析元信息
        meta_ret = parser.parse_meta()
        if isinstance(meta_ret, dict):
            # ✅ 如果像 IOPParser 一样直接返回 dict，就用它
            meta_data = meta_ret
        else:
            # ✅ 否则，从对象属性里自己组装一个 dict（兼容现在的 ElsevierParser 写法）
            meta_data = {
                "title": getattr(parser, "title", ""),
                "journal": getattr(parser, "journal", ""),
                "date": getattr(parser, "date", ""),
                "abstract": getattr(parser, "abstract", ""),
            }

        print(f"📄 Title: {meta_data.get('title', 'N/A')}")
        print(f"📘 Journal: {meta_data.get('journal', 'N/A')}")
        # print(f"🧾 Abstract: {meta_data.get('abstract', 'N/A')}")
        # print(f"📅 Date: {meta_data.get('date', 'N/A')}")

        # 3. 解析段落（ElsevierParser.parse_paragraphs 返回的是“纯文本列表”）
        paragraph_texts = parser.parse_paragraphs()  # list[str]

        print(f"📝 段落数: {len(paragraph_texts)}")

        # 4. 构建输出文件名
        base_name = os.path.basename(filepath)
        # 去掉 .html / .htm / .xhtml / .xml 等后缀
        for ext in [".html", ".htm", ".xhtml", ".xml"]:
            if base_name.lower().endswith(ext):
                base_name = base_name[: -len(ext)]
                break

        # 处理成安全文件名
        safe_name = "".join(c for c in base_name if c.isalnum() or c in (" ", "_", "-"))
        output_path = os.path.join(output_folder, f"{safe_name}.txt")

        # 5. 写入到 txt 文件
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(f"Title: {meta_data.get('title', '')}\n")
            f.write(f"Journal: {meta_data.get('journal', '')}\n")
            f.write(f"Date: {meta_data.get('date', '')}\n")
            f.write(f"Abstract: {meta_data.get('abstract', '')}\n\n")
            f.write("Paragraphs:\n")
            for para in paragraph_texts:
                f.write(para + "\n\n")

        print(f"✅ 已保存段落 → {output_path}\n")

        # 如果你想顺便测试表格解析，这里也可以加一句：
        # parser.parse_tables()

    except Exception as e:
        print(f"❌ 解析失败：{filepath}\n错误信息：{e}\n")


def batch_parse_elsevier_folder(input_folder, output_folder):
    """批量解析，支持自动跳过已处理的文件"""
    os.makedirs(output_folder, exist_ok=True)

    html_files = [
        f for f in os.listdir(input_folder)
        if f.lower().endswith((".html", ".htm", ".xhtml", ".xml"))
    ]

    if not html_files:
        print("⚠️ 未找到 HTML/XML 文件。")
        return

    print(f"🚀 开始增量处理，共 {len(html_files)} 篇文献...\n")

    for html_file in html_files:
        # --- 核心改进：在调用解析前先做判断 ---
        base_name = html_file
        for ext in [".html", ".htm", ".xhtml", ".xml"]:
            if base_name.lower().endswith(ext):
                base_name = base_name[: -len(ext)]
                break
        
        safe_name = "".join(c for c in base_name if c.isalnum() or c in (" ", "_", "-"))
        output_path = os.path.join(output_folder, f"{safe_name}.txt")

        # 判断：如果文件已存在且大小超过 100 字节（排除空文件/损坏文件），则跳过
        if os.path.exists(output_path) and os.path.getsize(output_path) > 100:
            # print(f"⏭️  已处理过，跳过: {safe_name}")
            continue
        # ------------------------------------

        file_path = os.path.join(input_folder, html_file)
        parse_elsevier_article(file_path, output_folder)

    print("\n🎯 全部任务处理完成！")


if __name__ == "__main__":
    # 👉 换成你的 Elsevier HTML/XML 文件夹路径
    input_folder = DATA_ROOT / "Elsevier" / "source"   # 📂 输入 HTML/XML 文件夹路径
    output_folder = DATA_ROOT / "Elsevier" / "txt"     # 📂 输出 TXT 文件夹路径

    batch_parse_elsevier_folder(input_folder, output_folder)
 

🚀 开始批量解析 Elsevier 文献，共 204 篇...

📄 Title: Biodegradable self-reporting nanocomposite films of poly(lactic acid) nanoparticles engineered by layer-by-layer assembly
📘 Journal: 
✅ Found 22 paragraphs.

[1] Help...

[2] PolymerVolume 51, Issue 18, 19 August 2010, Pages 4127-4139...

[3] Share...

[4] Cite...

[5] Scheme 1. Schematic representation for preparation of bare PLA (1) and cationic PLA–PEI nanoparticles (2) by precipitation of PLA polymer from acetone...

[6] Fig. 1. AFM topography (left) and phase (right) images of bare PLA (a) and cationic PLA–PEI (b) nanoparticles drop-cast from aqueous dispersions on a ...

[7] Fig. 2. 3D AFM views and cross-section analysis of bare PLA (a) and modified PLA–PEI (b) nanoparticles with their size histograms (c and d, respective...

[8] Scheme 2. Schematic representation of PLA nanoparticle assemblies through hydrogen-bonding (1) or ionic interactions (2)....

[9] Scheme 3. Chemical structures of poly(lactic acid) (PLA), poly(N-vinylpyrrolidone